# 41. Structured output, JSON, and safe tool calling

**Level:** Intermediate

**Prerequisites:** 04, 05, and 40: data validation, HTTP, prompt boundaries.

[Course index](00_START_HERE.ipynb) | [Previous](40_Prompts_Context_and_Model_Boundaries.ipynb) | [Next](42_Retrieval_Augmented_Generation.ipynb)

**Topics:** Structured output, JSON output, Function calling, Tool calling, Pydantic, Validation.

**Matching script:** `examples\41_Structured_Output_and_Tool_Calling.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

JSON output is parseable text; structured output additionally follows a declared schema.
Neither guarantees the content is factually true. Provider-constrained decoding can improve
format compliance, but your application must still validate types, ranges, permissions, and
business rules.

Tool calling means a model proposes a named operation and arguments. The application decides
whether to execute it, performs the action, and returns a result. A tool call is not Python
code that should be eval'd, and it does not grant the model authority to run arbitrary commands.

Use an allowlisted registry, strict argument schemas, narrow scopes, timeouts, output limits,
audit, and explicit approval for consequential side effects. Treat tool output as untrusted
input too. A valid function signature does not prove a requested target is authorized.

The example validates a mock model response with Pydantic v2 and invokes a read-only local
calculator. It is real schema validation and dispatch, but no hosted model generates the call.

## Mathematics or data contract

A safe dispatch boundary is proposed JSON -> schema validation -> allowlist/authorization -> bounded tool -> validated result.

## Important limitation

This is an offline read-only demonstration; production tool permissions cannot be replaced by a prompt.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Validate and execute one allowlisted local tool

**Purpose:** Make the application, not generated text, control execution.

### Read the code in this order
1. Declare a strict schema with extra fields forbidden.
2. Validate a mock tool request.
3. Dispatch through a dictionary of known callables.
4. Reject a malformed port count and verify the valid result.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
wrong argument type rejected
{'free': 36}
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class CapacityArguments(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    total: int = Field(ge=0, le=10000)
    used: int = Field(ge=0, le=10000)

class ToolCall(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    name: Literal["free_ports"]
    arguments: CapacityArguments

def free_ports(total, used):
    if used > total:
        raise ValueError("used exceeds total")
    return {"free": total - used}

registry = {"free_ports": free_ports}
call = ToolCall.model_validate_json('{"name":"free_ports","arguments":{"total":48,"used":12}}')
result = registry[call.name](**call.arguments.model_dump())
assert result == {"free": 36}
try:
    ToolCall.model_validate_json('{"name":"free_ports","arguments":{"total":"48","used":12}}')
except ValidationError:
    print("wrong argument type rejected")
else:
    raise AssertionError("wrong type accepted")
print(result)

wrong argument type rejected
{'free': 36}


### Interpret example 1

The result should demonstrate this purpose: **Make the application, not generated text, control execution.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Reject unknown tool names and cross-field inconsistency

**Purpose:** Show that syntax and schema checks do not replace semantic tool invariants.

### Read the code in this order
1. Restrict tool names with Literal and forbid extra fields.
2. Reject an unsupported operation.
3. Check a cross-field total/used relationship explicitly.
4. Verify no action occurs for rejected requests.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- `raise` reports invalid input or a failed requirement instead of fabricating success.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
requests rejected before execution: 2
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, ValidationError, model_validator

class Request(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    tool: Literal["free_ports"]
    total: int
    used: int

    @model_validator(mode="after")
    def valid_capacity(self):
        if not 0 <= self.used <= self.total <= 10000:
            raise ValueError("inconsistent capacity")
        return self

bad_inputs = [{"tool": "run_command", "total": 48, "used": 12},
              {"tool": "free_ports", "total": 10, "used": 12}]
rejected = 0
for payload in bad_inputs:
    try:
        Request.model_validate(payload)
    except ValidationError:
        rejected += 1
    else:
        raise AssertionError("invalid request accepted")
assert rejected == 2
print("requests rejected before execution:", rejected)

requests rejected before execution: 2


### Interpret example 2

The result should demonstrate this purpose: **Show that syntax and schema checks do not replace semantic tool invariants.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add an unknown tool and extra argument rejection test; keep all tools read-only.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Structured output` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A safe dispatch boundary is proposed JSON -> schema validation -> allowlist/authorization -> bounded tool -> validated result.
- Name the boundary of the demonstration: This is an offline read-only demonstration; production tool permissions cannot be replaced by a prompt.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://docs.pydantic.dev/latest/concepts/models/`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.